# Qwen3.8-Flash-Next NVFP4 on 2x B200 (TP2), vLLM

| Metric | Value |
|---|---|
| Output tok/s (c=128) | **5,255.8** (measured) |
| $/M output tokens (c=128) | **0.66** (measured, $6.25/B200-hour) |
| Boot | 1,091 s |
| Errors | 0 across 960 requests |

Status: **research preview**. c=1 and c=8 are pending. See
[`recipe.md`](recipe.md) for the full writeup.

This notebook is self-sufficient: cells 1-3 set up the pins and replay the
committed receipt (no GPU needed); the later cells give a Modal launch path
and a plain `vllm serve` + `curl`/`python` path for anyone with a 2x B200 box.

In [ ]:
# --- Status cell ---
EXPERIMENT = "qwen3.8-flash-next-b200-tp2-calibration"
RECEIPT_PATH = "receipts/calibration-1788689995.json"
LIVE = False  # this notebook replays a committed receipt; set True only if you re-run the sweep yourself

print(f"experiment : {EXPERIMENT}")
print(f"LIVE       : {LIVE}")
print("status     : measured (concurrency sweep c=16/32/64/128). c=1/c=8 are")
print("             untested (pending) -- no receipt exists for those rows.")

In [ ]:
# --- Pins ---
pins = {
    "image_digest": "vllm/vllm-openai@sha256:41d42cfabd3289f40fdca71b4a0fe290474880d20e9534a90c698eb78e3eecee",
    "image_note": "nightly 1970f3ed, 2026-09-06, x86_64, contains the PLE fix (commit d4d703c)",
    "model": "nvidia/Qwen3.8-Flash-Next-NVFP4",
    "model_revision": "fc694b54fb0174e0913e6adf86691ef85a4ead47",
    "quantization": "modelopt (NVFP4)",
    "topology": "TP2, 2x B200",
    "server_flags": [
        "--max-model-len 8192",
        "--gpu-memory-utilization 0.90",
        "--max-num-seqs 128",
        "--trust-remote-code",
        "--reasoning-parser qwen3",
    ],
    "speculative_decoding": "none -- vLLM PR #55513 (MTP for this model) was open, not merged, "
                            "as of this pin; NVIDIA's model card additionally requires "
                            "--enable-expert-parallel for MTP, untested here",
    "pricing_basis_usd_per_b200_hour": 6.25,
}
for k, v in pins.items():
    print(f"{k:28s}: {v}")

## Results -- calibration sweep (measured)

Concurrency sweep at `--max-model-len 8192`, `max_tokens=1024`,
`N = 4 x concurrency` requests per level, first user turn only, temperature
1.0 / top-p 0.95 / top-k 20, thinking disabled. This replays the committed
receipt; it does not re-run the sweep (see the Reproduce cells below to do
that on your own Modal account or B200 box).

In [ ]:
# --- Load and render the committed receipt ---
import json
from IPython.display import display, Markdown

with open(RECEIPT_PATH) as f:
    receipt = json.load(f)

print(f"boot_time_s       : {receipt['boot_time_s']:.1f}")
print(f"gpu_memory        : {receipt['gpu_memory']}")
print(f"tensor_parallel   : {receipt['tensor_parallel_size']}")
print(f"total_card_hours  : {receipt['total_card_hours_this_job']:.3f}")
print(f"estimated_cost_usd: {receipt['estimated_cost_this_job_usd']:.2f}")

def fmt(v, nd=1):
    return "untested (pending)" if v is None else f"{v:,.{nd}f}"

rows = []
for c_label in ["1", "8", "16", "32", "64", "128"]:
    entry = receipt["sweep"].get(c_label)
    if entry is None:
        rows.append([c_label, "untested (pending)"] + ["--"] * 6)
        continue
    rows.append([
        c_label,
        entry["n_requests"],
        fmt(entry["output_tok_s"]),
        fmt(entry["prompt_tok_s"]),
        fmt(entry["mean_resp_tokens"]),
        entry["truncations"],
        entry["n_errors"],
        fmt(entry["dollars_per_m_output_tokens"], 2),
    ])

headers = ["Concurrency", "Requests", "Output tok/s", "Prompt tok/s",
           "Mean resp tokens", "Truncations", "Errors", "$/M output tokens"]
lines = ["| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
for row in rows:
    lines.append("| " + " | ".join(str(c) for c in row) + " |")
display(Markdown("\n".join(lines)))

assert not LIVE, "commit this notebook with LIVE = False"

## Pitfalls (measured)

- **1x B200 (TP1) OOMs on `torch.compile` autotune, not on weight load.**
  The 124 GiB of NVFP4 weights boot fine on one card; FlashInfer autotune
  then tries a 95 GiB scratch tensor and the process OOMs before the server
  becomes healthy. Untested fix: `--no-enable-flashinfer-autotune`
  (**label: untested**).
- **`--max-model-len 8192` truncates multi-turn prompts.** With 4,096 output
  tokens and this context cap, 3.4% of PerfectBlend conversations fail with
  a "maximum context length" error. Use `--max-model-len 16384` for
  multi-turn workloads.
- **No MTP yet.** vLLM PR
  [#55513](https://github.com/vllm-project/vllm/pull/55513) was open, not
  merged, as of this pin.

## Workload evidence

96,583 PerfectBlend conversations were regenerated with this recipe at
concurrency 128 (temperature 1.0 / top-p 0.95 / top-k 20, thinking off,
`max_tokens=4096`) -- see [`modal_regen.py`](modal_regen.py).

## Reproduce -- Modal path

Requires a Modal account and access to a volume named `qwen38-drafter`
holding `prompts_2k.jsonl` (any JSONL of
`{"conversations":[{"role":"user","content":...}]}` rows works).
[`modal_calibrate.py`](modal_calibrate.py) is the exact script that produced
the receipt above (cleaned of any private identifiers).

In [ ]:
# 1x B200 attempt (expected to OOM per the pitfall above -- kept for reproducibility)
# modal run --detach modal_calibrate.py

# 2x B200 attempt (produces the receipt replayed above)
# modal run --detach modal_calibrate.py --gpu-count 2

# fetch the resulting receipt
# modal volume get qwen38-drafter calibration-<timestamp>.json ./
print("see modal_calibrate.py in this folder for the full script")

## Reproduce -- bare-metal / non-Modal path

On any 2x B200 host with Docker and the NVIDIA Container Toolkit:

In [ ]:
# Cell: launch the server (run in a separate terminal / background process)
launch_cmd = """
docker run --gpus '"device=0,1"' --rm -p 127.0.0.1:8000:8000 --ipc=host \\
  vllm/vllm-openai@sha256:41d42cfabd3289f40fdca71b4a0fe290474880d20e9534a90c698eb78e3eecee \\
  --model nvidia/Qwen3.8-Flash-Next-NVFP4 \\
  --revision fc694b54fb0174e0913e6adf86691ef85a4ead47 \\
  --quantization modelopt \\
  --max-model-len 8192 \\
  --gpu-memory-utilization 0.90 \\
  --max-num-seqs 128 \\
  --tensor-parallel-size 2 \\
  --trust-remote-code \\
  --reasoning-parser qwen3
"""
print(launch_cmd)

In [ ]:
# Cell: once the server is healthy (curl http://127.0.0.1:8000/health), drive
# the same concurrency sweep with plain requests -- no Modal or openai SDK needed.
import json, time, urllib.request

def one_request(prompt_text, max_tokens=1024):
    body = json.dumps({
        "model": "nvidia/Qwen3.8-Flash-Next-NVFP4",
        "messages": [{"role": "user", "content": prompt_text}],
        "temperature": 1.0, "top_p": 0.95, "max_tokens": max_tokens,
        "extra_body": {"top_k": 20, "chat_template_kwargs": {"enable_thinking": False}},
    }).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:8000/v1/chat/completions", data=body,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=120) as resp:
        out = json.loads(resp.read())
    return out, time.time() - t0

# Example single call (requires a live server -- not executed by CI/replay):
# resp, dt = one_request("What is the capital of France?")
# print(resp["choices"][0]["message"]["content"], dt)
print("define your own concurrency driver around one_request() for a full sweep")